# BirdCLEF+ 2026 — Inference & Submission (Stage 1 Optimized)

V8 推理优化。改动: 10s上下文窗口(单路) / 时间平滑 / 改进后处理。
流程: 声景文件 → 12×10s 窗口(2.5+5+2.5) → torchaudio mel → 模型预测 → 时间平滑 → 后处理 → submission.csv

In [ ]:
import subprocess, os, time
T0 = time.time()

os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'

FORCE_CPU = False
try:
    r = subprocess.run(['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
                       capture_output=True, text=True, timeout=10)
    gpu_caps = [float(x.strip()) for x in r.stdout.strip().split('\n') if x.strip()]
    min_cap = min(gpu_caps) if gpu_caps else 99.0
    print(f'GPU compute capabilities: {gpu_caps}')
    if min_cap < 7.0:
        print(f'SM {min_cap} < 7.0 (P100) — using CPU fallback')
        FORCE_CPU = True
except Exception:
    print('nvidia-smi failed, assuming compatible GPU')
print(f'[{time.time()-T0:.1f}s] GPU check done')

In [ ]:
import json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import timm
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

USE_AMP = torch.cuda.is_available() and not FORCE_CPU
if USE_AMP:
    try:
        from torch.amp import autocast
        AMP_DEVICE = 'cuda'
    except ImportError:
        from torch.cuda.amp import autocast
        AMP_DEVICE = None

DEVICE = torch.device('cpu') if FORCE_CPU else torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'[{time.time()-T0:.1f}s] PyTorch {torch.__version__}, Device: {DEVICE}')

In [ ]:
# ── 路径配置 ──
IS_KAGGLE = Path('/kaggle/input').exists()

def find_data_dir():
    if not IS_KAGGLE: return Path('../data/raw')
    for p in Path('/kaggle/input').iterdir():
        if (p / 'test_soundscapes').exists(): return p
        if p.is_dir():
            for sub in p.iterdir():
                if sub.is_dir() and (sub / 'test_soundscapes').exists(): return sub
    raise FileNotFoundError('test_soundscapes not found')

def find_model_dir():
    """Search for model weights in known Kaggle mount paths."""
    candidates = [
        Path('/kaggle/input/notebooks/montyeternity/birdclef2026-baseline-b0-training'),
        Path('/kaggle/input/birdclef2026-baseline-b0-training'),
        Path('/kaggle/input/birdclef2026-weights'),
    ]
    for c in candidates:
        if c.exists():
            pth = list(c.glob('best_fold*.pth'))
            if pth:
                print(f'[MODEL] Found {[f.name for f in pth]} in {c}')
                return c

    if IS_KAGGLE:
        nb_dir = Path('/kaggle/input/notebooks')
        if nb_dir.exists():
            for pth in nb_dir.rglob('best_fold*.pth'):
                print(f'[MODEL] Found {pth.name} in {pth.parent}')
                return pth.parent
        for pth in Path('/kaggle/input').rglob('best_fold*.pth'):
            print(f'[MODEL] Found {pth.name} in {pth.parent}')
            return pth.parent

    return Path('../models')

DATA_DIR = find_data_dir()
MODEL_DIR = find_model_dir()
OUTPUT_DIR = Path('/kaggle/working') if IS_KAGGLE else Path('../models')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Data: {DATA_DIR}')
print(f'Models: {MODEL_DIR}')
print(f'Model files: {list(MODEL_DIR.glob("best_fold*.pth"))}')

In [ ]:
# ── 常量 ──
SR = 32_000
CLIP_DURATION = 5.0
CLIP_SAMPLES = int(SR * CLIP_DURATION)
N_FFT = 1024
HOP_LENGTH = 320
N_MELS = 128
FMIN = 50
FMAX = 14000
NUM_CLASSES = 234

submission = pd.read_csv(DATA_DIR / 'sample_submission.csv', nrows=0)
SPECIES_COLS = [c for c in submission.columns if c != 'row_id']
taxonomy = pd.read_csv(DATA_DIR / 'taxonomy.csv')
TAX_MAP = dict(zip(taxonomy['primary_label'].astype(str), taxonomy['class_name']))
LABEL2IDX = {label: i for i, label in enumerate(SPECIES_COLS)}
print(f'Species: {len(SPECIES_COLS)}')

In [ ]:
# ── 音频处理（torchaudio，与训练一致） ──
MEL_TRANSFORM = torchaudio.transforms.MelSpectrogram(
    sample_rate=SR, n_fft=N_FFT, hop_length=HOP_LENGTH,
    n_mels=N_MELS, f_min=FMIN, f_max=FMAX, power=2.0)
AMP_TO_DB = torchaudio.transforms.AmplitudeToDB(stype='power', top_db=80)
RESAMPLERS = {}

def load_audio(path, sr=SR, duration=CLIP_DURATION, offset=0.0):
    tl = int(sr * duration)
    try:
        waveform, file_sr = torchaudio.load(path)
        if waveform.shape[0] > 1:
            waveform = waveform.mean(dim=0, keepdim=True)
        if file_sr != sr:
            if file_sr not in RESAMPLERS:
                RESAMPLERS[file_sr] = torchaudio.transforms.Resample(file_sr, sr)
            waveform = RESAMPLERS[file_sr](waveform)
        start = int(offset * sr)
        waveform = waveform[:, start:start + tl].squeeze(0)
    except Exception:
        waveform = torch.zeros(tl)
    if waveform.numel() == 0:
        waveform = torch.zeros(tl)
    if waveform.numel() < tl:
        reps = (tl // waveform.numel()) + 1
        waveform = waveform.repeat(reps)[:tl]
    return waveform[:tl]

def audio_to_melspec(waveform):
    mel = AMP_TO_DB(MEL_TRANSFORM(waveform.unsqueeze(0))).squeeze(0)
    return (mel - mel.mean()) / (mel.std() + 1e-6)

# 验证
test_wav = torch.randn(CLIP_SAMPLES)
test_mel = audio_to_melspec(test_wav)
print(f'Mel test: {test_mel.shape}')

In [ ]:
# ── 模型（与训练一致：基线 avg pool + 单层 FC） ──
class BirdCLEFB0(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            'tf_efficientnet_b0_ns', pretrained=False,
            in_chans=1, num_classes=0, global_pool='avg')
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(1280, NUM_CLASSES)

    def forward(self, mel):
        x = self.backbone(mel)
        x = self.dropout(x)
        return self.fc(x)

# 加载权重
models = []
for p in sorted(MODEL_DIR.glob('best_fold*.pth')):
    m = BirdCLEFB0().to(DEVICE)
    m.load_state_dict(torch.load(p, map_location=DEVICE, weights_only=True))
    m.eval()
    models.append(m)
    print(f'  Loaded: {p.name}')
print(f'[{time.time()-T0:.1f}s] Total models: {len(models)}')
if not models:
    raise FileNotFoundError(f'No model weights found in {MODEL_DIR}')

In [ ]:
# ── 推理配置 ──
CONTEXT_SECONDS = 2.5     # 每侧额外上下文
CONTEXT_SAMPLES = int(SR * CONTEXT_SECONDS)
FULL_WINDOW_SAMPLES = CONTEXT_SAMPLES + CLIP_SAMPLES + CONTEXT_SAMPLES  # 10s total

TEMPERATURE = 1.0         # 1.0 = no scaling; adjust on validation if needed
TEMPORAL_ALPHA = 0.3      # neighbor smoothing weight

TIME_PRIOR = {
    'Aves':     [0.2,0.2,0.3,0.5,0.9,1.0,1.0,0.8,0.6,0.4,0.3,0.3,
                 0.3,0.3,0.3,0.4,0.5,0.8,1.0,0.6,0.3,0.2,0.2,0.2],
    'Amphibia': [0.8,0.8,0.7,0.5,0.3,0.2,0.2,0.1,0.1,0.1,0.1,0.1,
                 0.1,0.1,0.1,0.1,0.2,0.4,0.8,1.0,1.0,1.0,0.9,0.9],
    'Insecta':  [0.5,0.5,0.5,0.8,0.6,0.4,1.0,1.0,0.4,0.3,0.3,0.3,
                 0.3,0.3,0.3,0.3,0.3,0.4,0.5,0.8,0.5,0.5,0.5,0.5],
}

cond_prob = np.zeros((NUM_CLASSES, NUM_CLASSES))
labels_file = DATA_DIR / 'train_soundscapes_labels.csv'
if labels_file.exists():
    labels_df = pd.read_csv(labels_file)
    for _, row in labels_df.iterrows():
        idxs = [LABEL2IDX[l.strip()] for l in str(row.get('primary_label','')).split(';')
                if l.strip() in LABEL2IDX]
        for a in idxs:
            for b in idxs:
                if a != b: cond_prob[a][b] += 1
    cond_prob /= (cond_prob.sum(axis=1, keepdims=True) + 1e-8)
    print(f'Co-occurrence: {(cond_prob > 0).sum()} non-zero entries')
else:
    print('No train_soundscapes_labels.csv, skipping co-occurrence')

def parse_hour(filename):
    parts = Path(filename).stem.split('_')
    if len(parts) >= 6:
        try: return int(parts[5][:2])
        except: pass
    return 12

def postprocess(preds, hour):
    p = preds.copy()
    # co-occurrence boost (lowered threshold for soundscape regime)
    detected = np.where(p > 0.05)[0]
    for a in detected:
        for b in range(len(p)):
            if cond_prob[a][b] > 0.3:
                p[b] = max(p[b], p[a] * cond_prob[a][b] * 0.3)
    # time prior (gentle)
    h = hour % 24
    for idx, col in enumerate(SPECIES_COLS):
        cls = TAX_MAP.get(col, '')
        if cls in TIME_PRIOR:
            f = TIME_PRIOR[cls][h]
            p[idx] = p[idx] * 0.85 + p[idx] * f * 0.15
    return np.clip(p, 0, 1)

In [ ]:
# ── 推理（10s 上下文 + 温度缩放 + 时间平滑） ──
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')
test_dir = DATA_DIR / 'test_soundscapes'
print(f'[{time.time()-T0:.1f}s] Sample submission: {len(sample_sub)} rows')

file_windows = {}
for rid in sample_sub['row_id']:
    parts = rid.rsplit('_', 1)
    stem, end_sec = parts[0], int(parts[1])
    if stem not in file_windows:
        file_windows[stem] = []
    file_windows[stem].append((rid, end_sec))
for stem in file_windows:
    file_windows[stem].sort(key=lambda x: x[1])
print(f'Unique soundscapes: {len(file_windows)}')

def extract_context_clip(full_wav, end_sec):
    """Extract 10s clip centered on the 5s target window (2.5s + 5s + 2.5s)."""
    target_start = (end_sec - 5) * SR
    ctx_start = max(0, target_start - CONTEXT_SAMPLES)
    ctx_end = target_start + CLIP_SAMPLES + CONTEXT_SAMPLES
    clip = full_wav[ctx_start:ctx_end]
    if clip.numel() < FULL_WINDOW_SAMPLES:
        clip = F.pad(clip, (0, FULL_WINDOW_SAMPLES - clip.numel()))
    return clip[:FULL_WINDOW_SAMPLES]

def model_inference(batch, temperature=TEMPERATURE):
    """Run inference with temperature scaling on logits."""
    fold_logits = []
    for model in models:
        with torch.no_grad():
            if USE_AMP:
                with autocast(AMP_DEVICE) if AMP_DEVICE else autocast():
                    logits = model(batch)
            else:
                logits = model(batch)
        fold_logits.append(logits.cpu())
    avg_logits = torch.stack(fold_logits).mean(dim=0)
    return torch.sigmoid(avg_logits / temperature).numpy()

pred_dict = {}
t_infer = time.time()
for stem, windows in tqdm(file_windows.items(), desc='Inference'):
    filepath = test_dir / f'{stem}.ogg'
    hour = parse_hour(stem)

    try:
        full_wav, file_sr = torchaudio.load(str(filepath))
        if full_wav.shape[0] > 1:
            full_wav = full_wav.mean(dim=0, keepdim=True)
        if file_sr != SR:
            if file_sr not in RESAMPLERS:
                RESAMPLERS[file_sr] = torchaudio.transforms.Resample(file_sr, SR)
            full_wav = RESAMPLERS[file_sr](full_wav)
        full_wav = full_wav.squeeze(0)
    except Exception:
        full_wav = torch.zeros(SR * 60)

    # --- 10s context window inference (single path for speed) ---
    mels = []
    rids = []
    for row_id, end_sec in windows:
        clip_ctx = extract_context_clip(full_wav, end_sec)
        mel = audio_to_melspec(clip_ctx)
        mels.append(mel.unsqueeze(0))
        rids.append(row_id)

    batch = torch.stack(mels).to(DEVICE)
    raw_preds = model_inference(batch)

    # --- Temporal smoothing across adjacent windows ---
    n_windows = len(windows)
    smoothed = raw_preds.copy()
    if n_windows > 1:
        alpha = TEMPORAL_ALPHA
        for i in range(n_windows):
            neighbors = []
            if i > 0: neighbors.append(raw_preds[i-1])
            if i < n_windows - 1: neighbors.append(raw_preds[i+1])
            if neighbors:
                neighbor_avg = np.mean(neighbors, axis=0)
                smoothed[i] = (1 - alpha) * raw_preds[i] + alpha * neighbor_avg

    for i, rid in enumerate(rids):
        pred_dict[rid] = postprocess(smoothed[i], hour)

print(f'[{time.time()-T0:.1f}s] Inference done ({time.time()-t_infer:.1f}s for {len(pred_dict)} windows)')

rows = []
for rid in sample_sub['row_id']:
    row = {'row_id': rid}
    preds = pred_dict.get(rid)
    if preds is not None:
        for i, col in enumerate(SPECIES_COLS):
            row[col] = float(preds[i])
    else:
        for col in SPECIES_COLS:
            row[col] = 0.0
    rows.append(row)

sub = pd.DataFrame(rows)
sub.to_csv(OUTPUT_DIR / 'submission.csv', index=False)
print(f'Submission: {len(sub)} rows, {len(sub.columns)} columns')
print(f'Saved to: {OUTPUT_DIR / "submission.csv"}')
sub.head()

In [ ]:
# ── 提交验证 + 统计 ──
pred_cols = [c for c in sub.columns if c != 'row_id']
pred_vals = sub[pred_cols].values
print(f'Rows: {len(sub)}, Cols: {len(sub.columns)}')
print(f'Pred range: [{pred_vals.min():.6f}, {pred_vals.max():.4f}]')
print(f'Pred mean: {pred_vals.mean():.6f}, median: {np.median(pred_vals):.6f}')
print(f'Pred > 0.01: {(pred_vals > 0.01).sum()} ({(pred_vals > 0.01).mean()*100:.2f}%)')
print(f'Pred > 0.05: {(pred_vals > 0.05).sum()} ({(pred_vals > 0.05).mean()*100:.2f}%)')
print(f'Pred > 0.50: {(pred_vals > 0.50).sum()} ({(pred_vals > 0.50).mean()*100:.2f}%)')
print(f'Config: TEMPERATURE={TEMPERATURE}, TEMPORAL_ALPHA={TEMPORAL_ALPHA}, CONTEXT={CONTEXT_SECONDS}s')
print(f'Total time: {time.time()-T0:.1f}s')